# 03 - Click-Assisted Labeling (SAM3 tracker propagation)

Stage L2, for classes where text prompts fail (woodenboard, dummychild) or need
box prompts (bicyclestand - thin structure). One click (or 2-corner box) per
sequence propagates across the whole approach.

Outputs live under `labels/clicks/<CLICK_RUN_TAG>/` (STABLE tag, not dated -
clicks accumulate across sessions and survive disconnects). Old clicks.json
files can be imported via `IMPORT_CLICKS_JSON`; frame dirs are re-derived from
the shared cache, so old absolute paths don't matter.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None   # set to redirect all artifact I/O (e.g. for tests)
RAW_ROOT_OVERRIDE = None    # set to point at a different raw-sequence root

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import base64
import json as _json
import random
import time
from collections import defaultdict

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from vision_uss_research.labeling.boxes_io import (append_boxes, done_keys,
                                                   relativize_frame_path)
from vision_uss_research.labeling.corridor import masks_to_bbox
from vision_uss_research.runs import resolve_run
from vision_uss_research.sequences import (extract_sequence_frames, frames_window_id,
                                           nearest_frame, select_camera_videos,
                                           with_drive_retry)

CLICK_RUN_TAG = "clicks_v1"          # stable across sessions
TARGET_OBJECTS = {"woodenboard": 100, "dummychild": 35, "bicyclestand": 35}
BOX_PROMPT_OBJECTS = {"bicyclestand"}
REDO_OBJECTS: set[str] = set()       # re-ask + re-propagate only these
RESET_CLICK_HISTORY = False
IMPORT_CLICKS_JSON = None            # path to an archived clicks.json to import
MANUAL_SEQUENCE_IDS: list[str] = []
N_FRAMES = 20
RATIO_RANGE = (0.30, 0.85)
CLICK_RATIO = 0.60
SEED = 0
INVENTORY_RUN = None                 # None = newest non-smoke inventory run

RUN_DIR = ARTIFACTS / "labels" / "clicks" / CLICK_RUN_TAG
FRAMES_ROOT = ARTIFACTS / "frames"
WINDOW_ROOT = FRAMES_ROOT / frames_window_id(N_FRAMES, RATIO_RANGE)
CLICKS_JSON = RUN_DIR / "clicks.json"
BOXES_CSV = RUN_DIR / "propagated_boxes.csv"
RUN_DIR.mkdir(parents=True, exist_ok=True)

inv_dir = resolve_run(ARTIFACTS / "inventory", "inventory", run_id=INVENTORY_RUN,
                      require_file="inventory_folders.csv", exclude_suffix="_smoke")
assert inv_dir is not None, "no inventory run found - run notebook 01 first"
print(f"click run: {RUN_DIR}\ninventory: {inv_dir.name}")

In [ ]:
def load_clicks() -> dict:
    return _json.loads(CLICKS_JSON.read_text()) if CLICKS_JSON.exists() else {}

def save_clicks(clicks: dict) -> None:
    CLICKS_JSON.write_text(_json.dumps(clicks, indent=1))

def item_key(item: dict) -> str:
    return f"{item['sequence_id']}::{item['camera']}"

if RESET_CLICK_HISTORY and CLICKS_JSON.exists():
    backup = CLICKS_JSON.with_name(f"clicks_backup_{time.strftime('%Y%m%d_%H%M%S')}.json")
    CLICKS_JSON.rename(backup)
    print(f"click history archived to {backup.name}")

if IMPORT_CLICKS_JSON:
    clicks = load_clicks()
    imported = _json.loads(Path(IMPORT_CLICKS_JSON).read_text())
    n_new = 0
    for key, entry in imported.items():
        if key not in clicks and (entry.get("point") or entry.get("box")):
            entry.pop("frames_dir", None)  # re-derived from the shared cache
            clicks[key] = entry
            n_new += 1
    save_clicks(clicks)
    print(f"imported {n_new} clicks from {IMPORT_CLICKS_JSON}")

In [ ]:
inv = pd.read_csv(inv_dir / "inventory_folders.csv")
usable = inv[(inv["status"] == "complete")
             & ((inv["has_front_video"] == 1) | (inv["has_rear_video"] == 1))]

rng = random.Random(SEED)
selected = []
if MANUAL_SEQUENCE_IDS:
    for seq_id in MANUAL_SEQUENCE_IDS:
        row = inv[inv["sequence_id"] == seq_id].iloc[0]
        objs = str(row["label_v2_objects"]).split("|")
        target = next((o for o in objs if o.lower() != "empty"), objs[0])
        selected.append({"sequence_id": seq_id, "target_object": target})
else:
    for obj, n_wanted in TARGET_OBJECTS.items():
        mask = usable["label_v2_objects"].fillna("").str.split("|").apply(
            lambda names: any(n.strip().lower() == obj for n in names))
        candidates = sorted(usable[mask]["sequence_id"].tolist())
        picks = rng.sample(candidates, min(n_wanted, len(candidates)))
        selected += [{"sequence_id": s, "target_object": obj} for s in picks]
        print(f"{obj:<15} {len(candidates):>5} candidates -> selected {len(picks)}")

# click items: extract frames (shared cache), one item per sequence x camera
click_items, skipped = [], []
for sel in tqdm(selected, desc="Preparing click items", unit="seq"):
    seq_dir = RAW_ROOT / sel["sequence_id"]
    try:
        if not with_drive_retry(seq_dir.is_dir):
            skipped.append({**sel, "error": "not_a_dir"})
            continue
        videos, _direction = with_drive_retry(select_camera_videos, seq_dir)
    except Exception as e:
        skipped.append({**sel, "error": f"{type(e).__name__}: {e}"})
        continue
    for camera, video_path in videos:
        try:
            frames_dir = with_drive_retry(extract_sequence_frames, FRAMES_ROOT,
                                          sel["sequence_id"], camera, video_path,
                                          N_FRAMES, RATIO_RANGE)
        except Exception as e:
            skipped.append({**sel, "camera": camera, "error": f"{type(e).__name__}: {e}"})
            continue
        if frames_dir is None:
            continue
        click_idx = int(round((CLICK_RATIO - RATIO_RANGE[0])
                              / (RATIO_RANGE[1] - RATIO_RANGE[0]) * (N_FRAMES - 1)))
        click_items.append({"sequence_id": sel["sequence_id"],
                            "target_object": sel["target_object"],
                            "camera": camera, "click_frame_idx": click_idx})

print(f"click items: {len(click_items)}, skipped: {len(skipped)}")
if skipped:
    pd.DataFrame(skipped).to_csv(RUN_DIR / "skipped_sequences.csv", index=False)

## Click round (resumable; saves after every answer)

Point classes: click the object, check the **zoomed magnifier**, re-click to
adjust, **Confirm**. Box classes (bicyclestand): click two opposite corners -
dashed rectangle previews the box; a third click restarts. **Skip** when the
object is not visible (wrong camera).

In [ ]:
def frames_dir_for(item: dict) -> Path:
    return WINDOW_ROOT / item["sequence_id"] / item["camera"]

def ask_click_colab(image_path: Path, title: str, mode: str = "point"):
    from google.colab.output import eval_js
    from IPython.display import display as _display, HTML

    b64 = base64.b64encode(image_path.read_bytes()).decode()
    uid = f"clk_{random.randrange(10**9)}"
    hint = ("draw a BOX: click one corner of the object"
            if mode == "box" else "click the object...")
    _display(HTML(f'''
      <div style="font-family:sans-serif;margin:4px 0"><b>{title}</b></div>
      <div style="position:relative;display:inline-block;line-height:0">
        <img id="{uid}" src="data:image/jpeg;base64,{b64}"
             style="max-width:100%;cursor:crosshair;border:2px solid #888"/>
        <div id="{uid}_mk" style="position:absolute;display:none;width:26px;height:26px;
             margin:-13px 0 0 -13px;pointer-events:none;border:3px solid lime;
             border-radius:50%;box-shadow:0 0 0 2px rgba(0,0,0,.6)">
          <div style="position:absolute;left:50%;top:50%;width:4px;height:4px;
               margin:-2px 0 0 -2px;background:red;border-radius:50%"></div>
        </div>
        <div id="{uid}_rect" style="position:absolute;display:none;pointer-events:none;
             border:3px dashed lime;box-shadow:0 0 0 1px rgba(0,0,0,.6)"></div>
      </div>
      <div style="display:flex;align-items:center;gap:14px;margin:6px 0">
        <div style="text-align:center;font-family:sans-serif;font-size:12px;color:#555">
          <canvas id="{uid}_zoom" width="180" height="180"
                  style="border:2px solid #888;background:#222;display:block"></canvas>
          zoomed check
        </div>
        <div>
          <div id="{uid}_st" style="font-family:sans-serif;margin:6px 0;color:#555">
            {hint}</div>
          <button id="{uid}_ok" disabled
                  style="padding:6px 20px;font-weight:bold">Confirm</button>
          <button id="{uid}_skip" style="padding:6px 16px;margin-left:8px">Skip</button>
        </div>
      </div>
    '''))
    coords = eval_js(f'''
      new Promise(resolve => {{
        const MODE = "{mode}";
        const img = document.getElementById("{uid}");
        const mk = document.getElementById("{uid}_mk");
        const rect = document.getElementById("{uid}_rect");
        const st = document.getElementById("{uid}_st");
        const ok = document.getElementById("{uid}_ok");
        const zoom = document.getElementById("{uid}_zoom");
        let pt = null, c1 = null, box = null;
        const drawZoom = (nx, ny) => {{
          const half = 30;
          const ctx = zoom.getContext("2d");
          ctx.imageSmoothingEnabled = false;
          ctx.clearRect(0, 0, zoom.width, zoom.height);
          ctx.drawImage(img, nx - half, ny - half, 2 * half, 2 * half,
                        0, 0, zoom.width, zoom.height);
          ctx.strokeStyle = "lime"; ctx.lineWidth = 1;
          ctx.beginPath();
          ctx.moveTo(zoom.width / 2, 0); ctx.lineTo(zoom.width / 2, zoom.height);
          ctx.moveTo(0, zoom.height / 2); ctx.lineTo(zoom.width, zoom.height / 2);
          ctx.stroke();
          ctx.strokeStyle = "red"; ctx.lineWidth = 2;
          ctx.beginPath();
          ctx.arc(zoom.width / 2, zoom.height / 2, 9, 0, 6.283);
          ctx.stroke();
        }};
        img.onclick = e => {{
          const r = img.getBoundingClientRect();
          const dx = e.clientX - r.left, dy = e.clientY - r.top;
          const nx = dx * img.naturalWidth / r.width;
          const ny = dy * img.naturalHeight / r.height;
          drawZoom(nx, ny);
          if (MODE === "point") {{
            pt = [nx, ny];
            mk.style.left = dx + "px"; mk.style.top = dy + "px";
            mk.style.display = "block";
            st.textContent = "selected (" + Math.round(nx) + ", " + Math.round(ny)
                           + ") - check the zoom, click again to adjust, then Confirm";
            st.style.color = "#0a0";
            ok.disabled = false;
          }} else if (!c1 || box) {{
            c1 = {{dx: dx, dy: dy, nx: nx, ny: ny}};
            box = null;
            rect.style.display = "none";
            mk.style.left = dx + "px"; mk.style.top = dy + "px";
            mk.style.display = "block";
            st.textContent = "corner 1 set (" + Math.round(nx) + ", " + Math.round(ny)
                           + ") - now click the OPPOSITE corner";
            st.style.color = "#c60";
            ok.disabled = true;
          }} else {{
            box = [Math.min(c1.nx, nx), Math.min(c1.ny, ny),
                   Math.max(c1.nx, nx), Math.max(c1.ny, ny)];
            rect.style.left = Math.min(c1.dx, dx) + "px";
            rect.style.top = Math.min(c1.dy, dy) + "px";
            rect.style.width = Math.abs(dx - c1.dx) + "px";
            rect.style.height = Math.abs(dy - c1.dy) + "px";
            rect.style.display = "block";
            mk.style.display = "none";
            st.textContent = "box set (" + box.map(Math.round).join(", ")
                           + ") - click again to restart, or Confirm";
            st.style.color = "#0a0";
            ok.disabled = false;
          }}
        }};
        ok.onclick = () => {{
          img.style.border = "4px solid lime";
          st.textContent = "confirmed";
          resolve(MODE === "point" ? pt : box);
        }};
        document.getElementById("{uid}_skip").onclick = () => resolve(null);
      }})
    ''')
    return tuple(coords) if coords else None

def ask_click_local(image_path: Path, title: str, mode: str = "point"):
    print(f"{title}\nimage: {image_path}")
    if mode == "box":
        raw = input("enter 'x0,y0,x1,y1' box coords, or blank to skip: ").strip()
        return tuple(map(float, raw.split(","))) if raw else None
    raw = input("enter 'x,y' pixel coords, or blank to skip: ").strip()
    if not raw:
        return None
    x, y = raw.split(",")
    return (float(x), float(y))

In [ ]:
clicks = load_clicks()
pending = [it for it in click_items
           if item_key(it) not in clicks or it["target_object"] in REDO_OBJECTS]
if REDO_OBJECTS:
    print(f"REDO_OBJECTS={sorted(REDO_OBJECTS)}: re-asking those")
print(f"already answered: {len(clicks)} | pending: {len(pending)}\n")

for i, item in enumerate(pending):
    image_path = nearest_frame(frames_dir_for(item), item["click_frame_idx"])
    if image_path is None:
        clicks[item_key(item)] = {**item, "point": None, "error": "no_frames_extracted"}
        save_clicks(clicks)
        continue
    actual_idx = int(image_path.stem)
    mode = "box" if item["target_object"] in BOX_PROMPT_OBJECTS else "point"
    action = ("draw a box around the WHOLE object (2 corner clicks)"
              if mode == "box" else "click the object")
    title = (f"[{i + 1}/{len(pending)}] {item['target_object']} - "
             f"{item['sequence_id']} ({item['camera']} camera) - {action}")
    try:
        result = (ask_click_colab(image_path, title, mode) if IN_COLAB
                  else ask_click_local(image_path, title, mode))
    except Exception as e:
        clicks[item_key(item)] = {**item, "point": None,
                                  "error": f"{type(e).__name__}: {e}"}
        save_clicks(clicks)
        continue
    entry = {**item, "click_frame_idx": actual_idx, "point": None, "box": None}
    if result is not None:
        entry["box" if len(result) == 4 else "point"] = list(result)
    clicks[item_key(item)] = entry
    save_clicks(clicks)

n_answered = sum(1 for v in clicks.values() if v.get("point") or v.get("box"))
print(f"click round done: {n_answered} annotated, {len(clicks) - n_answered} skipped")

## Propagation round (GPU; resumable - rerun any time)

In [ ]:
import torch

from vision_uss_research.labeling.boxes_io import load_boxes_tolerant
from vision_uss_research.labeling.sam3 import Sam3Engine

engine = Sam3Engine()
print(f"SAM3 tracker on {engine.device}")

done = done_keys(BOXES_CSV)
if REDO_OBJECTS and BOXES_CSV.exists():
    prev = load_boxes_tolerant(BOXES_CSV)
    redo_mask = prev["target_object"].isin(list(REDO_OBJECTS))
    if redo_mask.any():
        prev[~redo_mask].to_csv(BOXES_CSV, index=False)
        done = done_keys(BOXES_CSV)
        print(f"dropped {int(redo_mask.sum())} rows for REDO_OBJECTS")

clicks = load_clicks()
entries = [v for v in clicks.values()
           if (v.get("point") or v.get("box")) and item_key(v) not in done]
print(f"to propagate: {len(entries)}")

for j, entry in enumerate(entries):
    frames_dir = frames_dir_for(entry)
    prompt = {"point": entry.get("point"), "box": entry.get("box")}
    masks = None
    for attempt in range(2):
        try:
            masks = engine.propagate(frames_dir, entry["click_frame_idx"], prompt)
            break
        except torch.cuda.OutOfMemoryError:
            print(f"OOM {entry['sequence_id']} - retrying")
            engine.free()
        except Exception as e:
            print(f"FAILED {entry['sequence_id']}: {type(e).__name__}: {e}")
            break
    engine.free()
    if not masks:
        continue
    rows = []
    for frame_idx, mask in sorted(masks.items()):
        bbox = masks_to_bbox(mask)
        rows.append({
            "sequence_id": entry["sequence_id"], "target_object": entry["target_object"],
            "camera": entry["camera"], "frame_idx": frame_idx,
            "frame_path": relativize_frame_path(
                frames_dir / f"{frame_idx:05d}.jpg", WINDOW_ROOT),
            "x0": bbox[0] if bbox else None, "y0": bbox[1] if bbox else None,
            "x1": bbox[2] if bbox else None, "y1": bbox[3] if bbox else None,
            "mask_area_frac": float(mask.mean()),
        })
    append_boxes(BOXES_CSV, rows)
    found = sum(1 for r in rows if r["x0"] is not None)
    print(f"[{j + 1}/{len(entries)}] {entry['sequence_id']} "
          f"({entry['target_object']}): {found}/{len(rows)} frames with mask")

## Sanity gallery (full QA lives in notebook 04)

In [ ]:
from vision_uss_research.labeling.boxes_io import resolve_frame_path, row_bbox

boxes = load_boxes_tolerant(BOXES_CSV)
boxed = boxes[boxes["x0"].notna()]
sample = boxed.groupby("target_object").head(3)
fig, axes = plt.subplots(1, min(len(sample), 6), figsize=(24, 3.2), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    p = resolve_frame_path(row, WINDOW_ROOT)
    if p is None:
        continue
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    x0, y0, x1, y1 = row_bbox(row)
    cv2.rectangle(img, (x0, y0), (x1, y1), (0, 255, 0), 3)
    ax.imshow(img)
    ax.set_title(row["target_object"], fontsize=9)
plt.show()
print(f"click run boxes: {len(boxed)} -> labels/clicks/{CLICK_RUN_TAG}")
print("next: QA in notebook 04")